# Updating primer list with data from scanned IDT Spec Sheets

Run in sessons launched from https://github.com/fomightez/3Dscatter_plot_mod_playground-binder
    
----
    
----    


In [1]:
%pip install pytesseract pdf2image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pdf2image]
Note: you may need to restart the kernel to use updated packages.


In a terminal, run the following:

```
conda install -c conda-forge poppler tesseract -y
```

WHEN THAT COMPLETES, YOU CAN RUN...

In [2]:
!curl -OL https://raw.githubusercontent.com/fomightez/idt_spec_sheet_processing/refs/heads/main/scripts/idt_primer_spec_sheet_ocrOct29-2026.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 22898  100 22898    0     0  53768      0 --:--:-- --:--:-- --:--:-- 53877


In [3]:
!mv idt_primer_spec_sheet_ocrOct29-2026.py idt_primer_spec_sheet_ocr.py

The next code cell will iterate on any PDF files in the root directory, assume they are scans of spec sheets & mine them for data.
If there is a PDF content that isn't a spec sheet, the mining will probably fail and so it is moot. However, if you only want certain PDFs scanned, you'll need to point the script at each specifcially, like `%run idt_primer_spec_sheet_ocr.py a_few_IDT_primer_spec_sheets.pdf`.

Mine the spec sheets. (Each PDF of spec sheet will make a separate `.csv` file of the mined data.)

In [ ]:
from pathlib import Path
import re
def natural_key(text):
    """Sort so that primer_2 comes before primer_10, ignoring case."""
    return [int(t) if t.isdigit() else t.casefold() for t in re.split(r"(\d+)", str(text))]
current_dir = Path(".") 
pdf_files = sorted(
    (p for p in current_dir.rglob("*")
     if p.is_file() and p.suffix.lower() == ".pdf"
     and not any(part.startswith(".") for part in p.relative_to(current_dir).parts[:-1]) )  # skips .ipynb_checkpoints
,key=natural_key)
print(f"{len(pdf_files)} PDF file(s) found")

for pf in pdf_files:
    %run idt_primer_spec_sheet_ocr.py {pf}

2 PDF file(s) found
p1.1: Beta Actin Forward Primer | 25 bases | Tm 63.6 | OK (crop x2 psm7)
... 


Place the established primer list in the root directory of the session using the file browser that is probably on the left side.

With the information collected from the sheets & the established list provided, we just need to do some additional housekeeping to prepare for the combining script that boils the collected information down to just what is needed in the primer list.  
We'll create a directory and place any `.csv` files with the extracted data in that directory so that we can subsequently point the combining script at that directory.

In [1]:
!mkdir -p extracted_csvs

In [2]:
!mv *.csv extracted_csvs/

In [3]:
!curl -OL https://raw.githubusercontent.com/fomightez/idt_spec_sheet_processing/refs/heads/main/scripts/idt_primer_spec_sheets_combine.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 27876  100 27876    0     0  78411      0 --:--:-- --:--:-- --:--:-- 78523


Now take the mined data and put it into a 'boiled-down' form that is just the needed information from  the generated `.csv` files, respecting the main,established listing that already exists.

In [4]:
%run idt_primer_spec_sheets_combine.py --main draft_ESTABLISHED_primer_list.tsv --main-order extracted_csvs

Searched: /home/jovyan/extracted_csvs
CSV files used: 1   skipped: 0
Rows read: 20   exact duplicates removed (same name and sequence): 0   renamed duplicates merged (same sequence, different name): 0   final entries: 20
Status of final entries: OK=17, OK_LENGTH_ONLY=2, REVIEW=1
WARNING, name overrides that matched no row: '18s rRNAR'

==== Main file: /home/jovyan/draft_ESTABLISHED_primer_list.tsv
Main-file rows: 146   (header row found)
Spec-sheet entries already in the main file (ignored, main file wins): 0
NEW entries added: 20

Wrote 166 entries (146 from the main file + 20 new) to /home/jovyan/extracted_csvs/combined_primers.tsv
Wrote 20 new entries only to /home/jovyan/extracted_csvs/combined_primers_new_only.tsv


**Don't directly open the produced `.tsv` file with Excel.** You should open it with a text editor and verify it shows the degree symbols correctly and then copy-paste that into an new spreadsheet. Then the degree and other symbols won't come out garbled. (I suspect same holds for other Spreadsheet software, like Google Sheets.)